# IMPORTS

In [0]:
import requests
from pyspark.sql import SparkSession
from pyspark.sql.functions import regexp_extract_all, regexp_replace,col, lit, size,  explode,split, element_at,col,udf,length,make_date,expr,weekofyear,collect_list,explode,max,min,lag,log,avg,when,abs,count,std,stddev,isnan
from pyspark.sql.types import StringType
import json
from pyspark.sql import Window
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_selection import mutual_info_classif
from scipy.stats import pointbiserialr
import numpy as np
import seaborn as sns
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.types import NumericType
import datetime

In [0]:
spark = SparkSession.getActiveSession()
if spark is None:
    spark = SparkSession.builder.getOrCreate()

# Data read

In [0]:
prices = spark.read.format('delta').load('/Volumes/market-mood/processed/prices_clean') 
spotify = spark.read.format('delta').load('/Volumes/market-mood/processed/spotify_aggregated')
tweets = spark.read.format('delta').load('/Volumes/market-mood/processed/sentiment_scores')

In [0]:
# Rango de precios
prices.agg({"date": "min"}).show()
prices.agg({"date": "max"}).show()

# Rango de tweets
tweets.agg({"date": "min"}).show()
tweets.agg({"date": "max"}).show()

# Rango de Spotify
spotify.agg({"year": "min"}).show()
spotify.agg({"year": "max"}).show()

In [0]:
spotify.filter(col("year") == 2021).count()


In [0]:
prices.printSchema()
spotify.printSchema()
tweets.printSchema()

# Data join

In [0]:
years = spotify.select('year').distinct().collect()

In [0]:
years =[y['year'] for y in years]

In [0]:
dates = pd.DataFrame(columns=['year','date'])
for year in years:
    for month in range(1,13):
        for day in range(1,32):
            try:
                dates = pd.concat([dates,pd.DataFrame({'year':year,'date':datetime.date(2022,month,day)},index=[0])])
            except:
                pass
dates = spark.createDataFrame(dates)

dates = dates.withColumn('week',weekofyear(col('date')))


In [0]:
dates = dates.groupBy('year','week').agg(collect_list('date').alias('dayList'))#collect list hace una lista de los elementos del groupby, en este caso, las fechas agrupadas por semana y año

In [0]:
spotify = spotify.join(dates,on=['year','week'])

In [0]:
spotify.printSchema()

In [0]:
spotify = spotify.withColumn('date', explode(spotify['dayList']))

In [0]:
spotify_dates = [x['date'] for x in spotify.select('date').collect()]
tweets_dates = [x['date'] for x in tweets.select('date').collect()]
print(prices.filter(col('date').isin(spotify_dates)).count())
print(prices.filter(col('date').isin(tweets_dates)).count())


In [0]:
print(spotify.filter(col('date').isin(tweets_dates)).count())

In [0]:
prices_dates = [x['date'] for x in prices.select('date').collect()]

In [0]:
spotify = spotify.filter(col('date').isin(prices_dates))
tweets = tweets.filter(col('date').isin(prices_dates))


In [0]:
spotify.select(max('date')).show()
spotify.select(min('date')).show()
tweets.select(max('date')).show()
tweets.select(min('date')).show()

# Feature calculus for finance

In [0]:
prices.printSchema()

## Log return

In [0]:
prices=prices.orderBy('date')

In [0]:
#lag es para mirar n filas anteriores
#definir en la ventana como se va a ordenar y dividir el dataset
w = Window.partitionBy('ticker').orderBy('date')
prices = prices.withColumn('log_return',log((col("close")/lag("close",1).over(w))))

In [0]:
prices.select("ticker", "date", "close", "log_return").show(10)


## RSI

In [0]:
#lag es para mirar n filas anteriores
#definir en la ventana como se va a ordenar y dividir el dataset
w_1 = Window.partitionBy('ticker').orderBy('date')
prices = prices.withColumn('difference',(col("close")-lag("close",1).over(w_1)))
w = Window.partitionBy('ticker').orderBy('date').rowsBetween(-13,0)
prices = prices.withColumn('avg_gains',avg(when(col("difference")>0,col('difference'))).over(w))
#when es para aplicar un filtro sobre la columna en la que se aplica la función, se puede usar .otherwise como un else, para devolver otra cosa en caso de no cumplir la condición
prices = prices.withColumn('avg_loses',avg(when(col("difference")<0,col('difference'))).over(w))

prices = prices.withColumn('rsi',when(count('close').over(w)<14,None).otherwise(100-(100/(1+col('avg_gains')/abs(col('avg_loses'))))))#el when es para que el RSI se calcule a partir de 14 días, que es cuando hay 14 ganancias o pérdidas
prices = prices.drop('avg_gains','avg_loses','difference')
#rango de filas que voy a usar
prices.select('ticker','date','close','log_return','rsi').show(20)


## MACD

In [0]:
#lag es para mirar n filas anteriores
#definir en la ventana como se va a ordenar y dividir el dataset
w_12 = Window.partitionBy('ticker').orderBy('date').rowsBetween(-11,0)
w_26 = Window.partitionBy('ticker').orderBy('date').rowsBetween(-25,0)
prices = prices.withColumn('ma_12',when(count('close').over(w_12)<12,None).otherwise(avg(col('close')).over(w_12)))
prices = prices.withColumn('ma_26',when(count('close').over(w_26)<26,None).otherwise(avg(col('close')).over(w_26)))
prices = prices.withColumn('macd',when((col('ma_12').isNull()|col('ma_26').isNull()),None).otherwise(col('ma_12')-col('ma_26')))

#rango de filas que voy a usar
prices.select('ticker','date','close','ma_12','ma_26','macd').show(30)

## Bollinger Bands

In [0]:
#
w_20 = Window.partitionBy('ticker').orderBy('date').rowsBetween(-19,0)
prices = prices.withColumn('std_20',when(count('close').over(w_20)<20,None).otherwise(std(col('close')).over(w_20)))
prices = prices.withColumn('bb_mid',when(count('close').over(w_20)<20,None).otherwise(avg(col('close')).over(w_20)))
prices = prices.withColumn('bb_upper',col('bb_mid')+2*col('std_20'))
prices = prices.withColumn('bb_lower',col('bb_mid')-2*col('std_20'))
prices = prices.withColumn('bb_position',when((col('bb_upper').isNull()|col('bb_lower').isNull()),None).otherwise((col('close')-col('bb_lower'))/(col('bb_upper')-col('bb_lower'))))



#rango de filas que voy a usar
prices.select('ticker','date','close','bb_upper','bb_lower','bb_position').show(30)


## Historic volatility

In [0]:
prices = prices.withColumn('volatility',when(count('close').over(w_20)<20,None).otherwise(stddev(col('log_return')).over(w_20)))

## Volume ratio

In [0]:
w_10 =  Window.partitionBy('ticker').orderBy('date').rowsBetween(-9,0)
prices = prices.withColumn('vol_ratio',when(count('close').over(w_10)<10,None).otherwise(col('volume')/avg('volume').over(w_10)))

In [0]:
prices.select('ticker','date','close','vol_ratio','volatility').show(30)

## Target

In [0]:
prices = prices.withColumn('difference',(col("close")-lag("close",1).over(w_1)))
prices = prices.withColumn('target',when(col('difference')>0,1).otherwise(0))
prices = prices.drop('difference')
prices.select('ticker','date','close','target').show(30)


# Load

In [0]:
feature_store_technical = prices.select(
    "ticker", "date", "year", "week",
    "close", "volume",
    "log_return", "rsi", "macd", "bb_position", 
    "volatility", "vol_ratio",
    "target"
)

feature_store_technical.printSchema()
feature_store_technical.count()

In [0]:
(feature_store_technical
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("ticker", "year")
    .save("/Volumes/market-mood/features/feature_store/"))

# Feature exploration

In [0]:
prices.select("ticker").distinct().show()

In [0]:
nvda = prices.filter(prices['ticker'] == 'NVDA')

In [0]:
nvda.select([count(when(col(c).isNull(), c)).alias(c) for c in nvda.columns]).show()

los nulos tienen sentido porqeu son valores que de penden de n días anteriores, por lo que si ni tienen esos n días, no
calculan y dan un nulo


In [0]:
nvda.printSchema()

In [0]:
type(nvda)

In [0]:
nvda_pd = nvda.toPandas()
nvda_pd=nvda_pd[nvda_pd['year']>=2024]
plt.figure(figsize=(15,10))
plt.plot(nvda_pd['date'],nvda_pd['close'],label='close')
plt.plot(nvda_pd['date'],nvda_pd['bb_upper'],label='bb_upper')
plt.plot(nvda_pd['date'],nvda_pd['bb_lower'],label='bb_lower')
plt.plot(nvda_pd['date'],nvda_pd['bb_mid'])
plt.plot(nvda_pd['date'],nvda_pd['volatility'],label='volatility')
plt.plot(nvda_pd['date'],nvda_pd['rsi'],label='rsi')
plt.legend()
plt.show()

In [0]:
nvda_target_count = nvda_pd.groupby('target')['date'].count().reset_index()
plt.pie(nvda_target_count['date'],labels=nvda_target_count['target'].unique())

# Correlaciones y relación entre variables

In [0]:
numeric_cols = [c.name for c in feature_store_technical.schema if isinstance(c.dataType,NumericType) and c.name != 'target']

In [0]:
assembler = VectorAssembler(
    inputCols=numeric_cols+['target'],
    outputCol='features',
    handleInvalid='skip')#saltar los nulos
df_vector = assembler.transform(feature_store_technical).select('features')


In [0]:
corr_mat = Correlation.corr(df_vector,'features',method='pearson')
corr_mat = corr_mat.collect()[0]['pearson(features)'].toArray()
display(corr_mat[-1])
plt.figure(figsize=(15,10))




In [0]:
sns.heatmap(np.asarray(corr_mat[-1]).reshape(11,1),yticklabels=numeric_cols, annot=True)

In [0]:
# Mutual Information
f_pd = feature_store_technical.toPandas().dropna()
mi = mutual_info_classif(f_pd[numeric_cols], f_pd["target"])
for col, score in zip(numeric_cols, mi):
    print(f"MI  {col}: {score:.4f}")

In [0]:
for col in numeric_cols:
    corr, pval = pointbiserialr(f_pd[col], f_pd["target"])
    print(f"PB  {col}: {corr:.4f} (p={pval:.4f})")